In [1]:
# =============================================================================
# MODULE 1: DATA LAYER
# CME SPAN Dummy Model — Contract Definitions, Portfolio, SPAN Parameters
# =============================================================================

import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

# =============================================================================
# 1.1 CONTRACT DEFINITIONS
# Each contract has a product code, expiry, tick size, and point value
# =============================================================================

@dataclass
class ContractSpec:
    product_code: str        # e.g. 'CL', 'NG', 'GC'
    product_name: str
    expiry: str              # e.g. 'Mar-25'
    contract_id: str         # unique: product_code + expiry
    tick_size: float         # minimum price move
    point_value: float       # dollar value of 1 full point move
    currency: str
    product_group: str       # for inter-commodity spreading

CONTRACT_SPECS: Dict[str, ContractSpec] = {
    "CL_Mar25": ContractSpec(
        product_code="CL",
        product_name="WTI Crude Oil",
        expiry="Mar-25",
        contract_id="CL_Mar25",
        tick_size=0.01,
        point_value=1000.0,    # $1000 per $1/bbl move (1000 bbl/contract)
        currency="USD",
        product_group="ENERGY"
    ),
    "CL_Jun25": ContractSpec(
        product_code="CL",
        product_name="WTI Crude Oil",
        expiry="Jun-25",
        contract_id="CL_Jun25",
        tick_size=0.01,
        point_value=1000.0,
        currency="USD",
        product_group="ENERGY"
    ),
    "NG_Mar25": ContractSpec(
        product_code="NG",
        product_name="Natural Gas",
        expiry="Mar-25",
        contract_id="NG_Mar25",
        tick_size=0.001,
        point_value=10000.0,   # $10,000 per $1/MMBtu move (10,000 MMBtu/contract)
        currency="USD",
        product_group="ENERGY"
    ),
    "GC_Apr25": ContractSpec(
        product_code="GC",
        product_name="Gold",
        expiry="Apr-25",
        contract_id="GC_Apr25",
        tick_size=0.10,
        point_value=100.0,     # $100 per $1/oz move (100 oz/contract)
        currency="USD",
        product_group="METALS"
    ),
}

# =============================================================================
# 1.2 PORTFOLIO POSITIONS
# net_position: +ve = long, -ve = short
# entry_price: used for unrealized P&L reference (not needed for SPAN margin)
# =============================================================================

@dataclass
class Position:
    contract_id: str
    net_position: int        # number of lots, signed
    entry_price: float       # for reference only
    current_price: float     # settlement price used in scenario generation

PORTFOLIO: Dict[str, Position] = {
    "CL_Mar25": Position(
        contract_id="CL_Mar25",
        net_position=+5,       # long 5 lots
        entry_price=78.50,
        current_price=79.20
    ),
    "CL_Jun25": Position(
        contract_id="CL_Jun25",
        net_position=-3,       # short 3 lots
        entry_price=79.80,
        current_price=80.10
    ),
    "NG_Mar25": Position(
        contract_id="NG_Mar25",
        net_position=+4,       # long 4 lots
        entry_price=2.650,
        current_price=2.710
    ),
    "GC_Apr25": Position(
        contract_id="GC_Apr25",
        net_position=+2,       # long 2 lots
        entry_price=2010.0,
        current_price=2025.0
    ),
}

# =============================================================================
# 1.3 SPAN PARAMETERS
# Scan range: max price move assumed (in price units, not %)
# SPAN uses 16 scenarios: 7 price move levels (±1..±3 sigma + extreme)
# combined with vol up/down — for futures, vol shift has no P&L impact
# so the 16 scenarios collapse to price moves only (vol shift ignored for futures)
#
# Scenario grid (CME standard):
#   Price moves: -3, -2, -1, 0, +1, +2, +3 scan range fractions
#                + 2 extreme scenarios at ±price_scan_range * 3 (33% weight)
# =============================================================================

@dataclass
class SPANParams:
    product_code: str
    price_scan_range: float      # max 1-day price move (absolute, in price units)
    # Intra-commodity spread charge (per spread, in $)
    # Applied to calendar spreads within same product
    intracommodity_spread_charge: float
    # Inter-commodity spread credit ratio (0 to 1)
    # How much of the smaller-side scan risk is credited back
    intercommodity_credit_rate: float

SPAN_PARAMS: Dict[str, SPANParams] = {
    "CL": SPANParams(
        product_code="CL",
        price_scan_range=4.00,           # $4/bbl max scan move for WTI
        intracommodity_spread_charge=200.0,   # $200 per calendar spread pair
        intercommodity_credit_rate=0.50       # 50% credit vs NG (correlated energy)
    ),
    "NG": SPANParams(
        product_code="NG",
        price_scan_range=0.50,           # $0.50/MMBtu max scan move
        intracommodity_spread_charge=150.0,
        intercommodity_credit_rate=0.50       # 50% credit vs CL
    ),
    "GC": SPANParams(
        product_code="GC",
        price_scan_range=35.00,          # $35/oz max scan move for Gold
        intracommodity_spread_charge=100.0,
        intercommodity_credit_rate=0.00       # no credit — metals vs energy
    ),
}

# =============================================================================
# 1.4 INTER-COMMODITY SPREAD TABLE
# Defines which product pairs get spreading credits and at what priority
# credit_rate: fraction of the smaller scan risk credited back
# =============================================================================

@dataclass
class InterCommoditySpread:
    leg1_product: str
    leg2_product: str
    credit_rate: float       # e.g. 0.50 = 50% of smaller scan risk is credited

INTERCOMMODITY_SPREADS: List[InterCommoditySpread] = [
    InterCommoditySpread(leg1_product="CL", leg2_product="NG", credit_rate=0.50),
]

# =============================================================================
# 1.5 SCENARIO DEFINITIONS
# Standard CME SPAN 16-scenario grid
# Each scenario is (price_fraction, vol_shift_direction, weight)
# price_fraction: multiplier on price_scan_range
# For futures, vol_shift has no effect — included for completeness/options later
# weight: used only for extreme scenarios (scenarios 15-16 = 35% weight)
# =============================================================================

SCENARIO_GRID: List[Tuple[float, str, float]] = [
    # (price_fraction, vol_shift, scenario_weight)
    (+1/3,  "up",   1.0),   # Scenario 1
    (+1/3,  "down", 1.0),   # Scenario 2
    (+2/3,  "up",   1.0),   # Scenario 3
    (+2/3,  "down", 1.0),   # Scenario 4
    (+1.0,  "up",   1.0),   # Scenario 5
    (+1.0,  "down", 1.0),   # Scenario 6
    (+2.0,  "up",   1.0),   # Scenario 7  [2x scan range]
    (+2.0,  "down", 1.0),   # Scenario 8
    (-1/3,  "up",   1.0),   # Scenario 9
    (-1/3,  "down", 1.0),   # Scenario 10
    (-2/3,  "up",   1.0),   # Scenario 11
    (-2/3,  "down", 1.0),   # Scenario 12
    (-1.0,  "up",   1.0),   # Scenario 13
    (-1.0,  "down", 1.0),   # Scenario 14
    (+2.0,  "up",   0.35),  # Scenario 15 — extreme up (35% weight)
    (-2.0,  "down", 0.35),  # Scenario 16 — extreme down (35% weight)
]

# =============================================================================
# 1.6 SANITY CHECK — print summary on load
# =============================================================================

def print_portfolio_summary():
    print("=" * 65)
    print("SPAN DUMMY MODEL — PORTFOLIO SUMMARY")
    print("=" * 65)
    print(f"\n{'Contract':<12} {'Direction':<10} {'Lots':<6} {'Curr Price':<12} {'Point Value':<12} {'Product Group'}")
    print("-" * 65)
    for cid, pos in PORTFOLIO.items():
        spec = CONTRACT_SPECS[cid]
        direction = "LONG" if pos.net_position > 0 else "SHORT"
        print(f"{cid:<12} {direction:<10} {abs(pos.net_position):<6} "
              f"{pos.current_price:<12.3f} {spec.point_value:<12.0f} {spec.product_group}")
    print("\n")
    print(f"{'Product':<8} {'Scan Range':<14} {'Intra Spread $':<18} {'Inter Credit %'}")
    print("-" * 55)
    for code, params in SPAN_PARAMS.items():
        print(f"{code:<8} {params.price_scan_range:<14.3f} "
              f"{params.intracommodity_spread_charge:<18.0f} "
              f"{params.intercommodity_credit_rate*100:.0f}%")
    print("=" * 65)

if __name__ == "__main__":
    print_portfolio_summary()

SPAN DUMMY MODEL — PORTFOLIO SUMMARY

Contract     Direction  Lots   Curr Price   Point Value  Product Group
-----------------------------------------------------------------
CL_Mar25     LONG       5      79.200       1000         ENERGY
CL_Jun25     SHORT      3      80.100       1000         ENERGY
NG_Mar25     LONG       4      2.710        10000        ENERGY
GC_Apr25     LONG       2      2025.000     100          METALS


Product  Scan Range     Intra Spread $     Inter Credit %
-------------------------------------------------------
CL       4.000          200                50%
NG       0.500          150                50%
GC       35.000         100                0%


In [2]:
# =============================================================================
# MODULE 2: RISK ARRAY ENGINE
# Generates 16-scenario P&L per contract, computes scan risk per position
# =============================================================================

# Paste Module 1 above this, or import from it
# from module1 import CONTRACT_SPECS, PORTFOLIO, SPAN_PARAMS, SCENARIO_GRID

import pandas as pd
import numpy as np

# =============================================================================
# 2.1 SCENARIO P&L FOR A SINGLE CONTRACT POSITION
# For futures: P&L = price_move * point_value * net_position
# price_move = price_fraction * price_scan_range
# weight applied only to extreme scenarios (15, 16)
# =============================================================================

def compute_scenario_pnl(contract_id: str) -> pd.DataFrame:
    """
    Returns a DataFrame with 16 rows (one per scenario) showing:
    - scenario number
    - price fraction and vol shift
    - price move (absolute, in price units)
    - hypothetical price
    - raw P&L (before weight)
    - weighted P&L (what SPAN actually uses)
    """
    pos   = PORTFOLIO[contract_id]
    spec  = CONTRACT_SPECS[contract_id]
    params = SPAN_PARAMS[spec.product_code]

    rows = []
    for i, (price_frac, vol_shift, weight) in enumerate(SCENARIO_GRID, start=1):
        price_move      = price_frac * params.price_scan_range
        hypo_price      = pos.current_price + price_move
        raw_pnl         = price_move * spec.point_value * pos.net_position
        weighted_pnl    = raw_pnl * weight          # extreme scenarios discounted to 35%

        rows.append({
            "Scenario"      : i,
            "Price_Frac"    : price_frac,
            "Vol_Shift"     : vol_shift,
            "Weight"        : weight,
            "Price_Move"    : round(price_move, 4),
            "Hypo_Price"    : round(hypo_price, 4),
            "Raw_PnL"       : round(raw_pnl, 2),
            "Weighted_PnL"  : round(weighted_pnl, 2),
        })

    return pd.DataFrame(rows)


# =============================================================================
# 2.2 SCAN RISK FOR A SINGLE POSITION
# Scan risk = max LOSS across all 16 weighted scenarios
# Loss = negative P&L, so scan risk = -min(weighted_pnl)
# Floor at 0 (can't have negative scan risk)
# =============================================================================

def compute_scan_risk(contract_id: str) -> dict:
    """
    Returns scan risk and the worst-case scenario details for a position.
    """
    df = compute_scenario_pnl(contract_id)

    worst_idx       = df["Weighted_PnL"].idxmin()
    worst_scenario  = df.loc[worst_idx]
    scan_risk       = max(0.0, -worst_scenario["Weighted_PnL"])

    return {
        "contract_id"       : contract_id,
        "net_position"      : PORTFOLIO[contract_id].net_position,
        "scan_risk"         : round(scan_risk, 2),
        "worst_scenario"    : int(worst_scenario["Scenario"]),
        "worst_price_move"  : worst_scenario["Price_Move"],
        "worst_pnl"         : worst_scenario["Weighted_PnL"],
    }


# =============================================================================
# 2.3 BUILD FULL RISK ARRAY FOR ALL POSITIONS
# =============================================================================

def build_risk_array() -> pd.DataFrame:
    """
    Returns a wide DataFrame: contracts as rows, scenarios as columns.
    Values are weighted P&L per scenario.
    """
    records = {}
    for cid in PORTFOLIO:
        df = compute_scenario_pnl(cid)
        records[cid] = df.set_index("Scenario")["Weighted_PnL"].to_dict()

    risk_array = pd.DataFrame(records).T
    risk_array.index.name   = "Contract"
    risk_array.columns      = [f"S{c}" for c in risk_array.columns]
    return risk_array


# =============================================================================
# 2.4 SCAN RISK SUMMARY TABLE
# =============================================================================

def build_scan_risk_summary() -> pd.DataFrame:
    rows = [compute_scan_risk(cid) for cid in PORTFOLIO]
    df = pd.DataFrame(rows)
    df = df.rename(columns={
        "contract_id"       : "Contract",
        "net_position"      : "Net_Pos",
        "scan_risk"         : "Scan_Risk ($)",
        "worst_scenario"    : "Worst_Scenario",
        "worst_price_move"  : "Worst_Price_Move",
        "worst_pnl"         : "Worst_PnL ($)",
    })
    return df


# =============================================================================
# 2.5 PRINT OUTPUTS
# =============================================================================

def print_scenario_pnl_table(contract_id: str):
    df = compute_scenario_pnl(contract_id)
    pos  = PORTFOLIO[contract_id]
    spec = CONTRACT_SPECS[contract_id]
    direction = "LONG" if pos.net_position > 0 else "SHORT"

    print(f"\n{'='*72}")
    print(f"SCENARIO P&L TABLE — {contract_id}  |  {direction} {abs(pos.net_position)} lots  "
          f"|  Settlement: {pos.current_price}")
    print(f"{'='*72}")
    print(f"{'Scen':<6} {'PriceFrac':>10} {'VolShift':>10} {'Weight':>7} "
          f"{'PriceMove':>11} {'HypoPrice':>11} {'RawPnL':>12} {'WtdPnL':>12}")
    print("-" * 72)
    for _, row in df.iterrows():
        print(f"{int(row.Scenario):<6} {row.Price_Frac:>10.4f} {row.Vol_Shift:>10} "
              f"{row.Weight:>7.2f} {row.Price_Move:>11.4f} {row.Hypo_Price:>11.4f} "
              f"{row.Raw_PnL:>12,.2f} {row.Weighted_PnL:>12,.2f}")
    print(f"{'='*72}")


def print_risk_array(risk_array: pd.DataFrame):
    print(f"\n{'='*72}")
    print("FULL RISK ARRAY — Weighted P&L per Scenario (all contracts)")
    print(f"{'='*72}")
    print(risk_array.to_string())
    print(f"{'='*72}")


def print_scan_risk_summary(summary: pd.DataFrame):
    print(f"\n{'='*72}")
    print("SCAN RISK SUMMARY — Worst-Case Loss per Position")
    print(f"{'='*72}")
    print(summary.to_string(index=False))
    print(f"Total Gross Scan Risk: ${summary['Scan_Risk ($)'].sum():>12,.2f}")
    print(f"{'='*72}")


# =============================================================================
# 2.6 RUN
# =============================================================================

if __name__ == "__main__":

    # --- Scenario P&L table for each contract ---
    for cid in PORTFOLIO:
        print_scenario_pnl_table(cid)

    # --- Full risk array (wide format) ---
    risk_array = build_risk_array()
    print_risk_array(risk_array)

    # --- Scan risk summary ---
    summary = build_scan_risk_summary()
    print_scan_risk_summary(summary)


SCENARIO P&L TABLE — CL_Mar25  |  LONG 5 lots  |  Settlement: 79.2
Scen    PriceFrac   VolShift  Weight   PriceMove   HypoPrice       RawPnL       WtdPnL
------------------------------------------------------------------------
1          0.3333         up    1.00      1.3333     80.5333     6,666.67     6,666.67
2          0.3333       down    1.00      1.3333     80.5333     6,666.67     6,666.67
3          0.6667         up    1.00      2.6667     81.8667    13,333.33    13,333.33
4          0.6667       down    1.00      2.6667     81.8667    13,333.33    13,333.33
5          1.0000         up    1.00      4.0000     83.2000    20,000.00    20,000.00
6          1.0000       down    1.00      4.0000     83.2000    20,000.00    20,000.00
7          2.0000         up    1.00      8.0000     87.2000    40,000.00    40,000.00
8          2.0000       down    1.00      8.0000     87.2000    40,000.00    40,000.00
9         -0.3333         up    1.00     -1.3333     77.8667    -6,666.67   

In [3]:
# =============================================================================
# MODULE 3: SPREADING MODULE
# Intracommodity (calendar) credits + Intercommodity credits
# =============================================================================

# Paste Modules 1 and 2 above this before running
# =============================================================================
# 3.1 INTRACOMMODITY (CALENDAR) SPREAD CREDIT
#
# Logic:
# - Find all positions within the same product code (e.g. CL Mar vs CL Jun)
# - A calendar spread = min(|long lots|, |short lots|) paired contracts
# - For each spread pair: credit = scan_risk_per_lot (long) + scan_risk_per_lot (short)
#   i.e. the natural offset in P&L from holding opposite legs
# - Charge: exchange levies a small intracommodity spread charge per spread
#   to prevent margin gaming via fake spreads
#
# Net intra credit = (offsetting P&L relief) - (spread charge * num_spreads)
# In SPAN: intra credit = sum of per-lot scan risks of paired legs
# =============================================================================

def compute_intracommodity_credits(scan_risk_summary: pd.DataFrame) -> dict:
    """
    Detects calendar spread pairs within the same product code.
    Returns credit breakdown per product.
    """
    results = {}

    # Group positions by product code
    scan_df = scan_risk_summary.copy()
    scan_df["product_code"] = scan_df["Contract"].apply(
        lambda x: CONTRACT_SPECS[x].product_code
    )
    scan_df["net_pos"] = scan_df["Contract"].apply(
        lambda x: PORTFOLIO[x].net_position
    )
    scan_df["lots"] = scan_df["net_pos"].abs()
    scan_df["scan_risk_per_lot"] = scan_df["Scan_Risk ($)"] / scan_df["lots"]

    for product, group in scan_df.groupby("product_code"):
        longs  = group[group["net_pos"] > 0]
        shorts = group[group["net_pos"] < 0]

        if longs.empty or shorts.empty:
            continue  # no calendar spread possible

        total_long_lots  = longs["lots"].sum()
        total_short_lots = shorts["lots"].sum()
        num_spreads      = int(min(total_long_lots, total_short_lots))

        if num_spreads == 0:
            continue

        params = SPAN_PARAMS[product]

        # Credit = per-lot scan risk of each leg × number of spread pairs
        # Use average per-lot scan risk across legs (weighted if multiple contracts)
        avg_long_risk_per_lot  = (longs["Scan_Risk ($)"].sum()  / total_long_lots)
        avg_short_risk_per_lot = (shorts["Scan_Risk ($)"].sum() / total_short_lots)

        gross_credit = (avg_long_risk_per_lot + avg_short_risk_per_lot) * num_spreads
        spread_charge = params.intracommodity_spread_charge * num_spreads
        net_credit   = gross_credit - spread_charge

        results[product] = {
            "product"               : product,
            "num_spreads"           : num_spreads,
            "long_contracts"        : list(longs["Contract"]),
            "short_contracts"       : list(shorts["Contract"]),
            "avg_long_risk_per_lot" : round(avg_long_risk_per_lot, 2),
            "avg_short_risk_per_lot": round(avg_short_risk_per_lot, 2),
            "gross_credit"          : round(gross_credit, 2),
            "spread_charge"         : round(spread_charge, 2),
            "net_intra_credit"      : round(net_credit, 2),
        }

    return results


# =============================================================================
# 3.2 INTERCOMMODITY SPREAD CREDIT
#
# Logic:
# - For each defined inter-commodity pair (e.g. CL vs NG):
#   - Take the total scan risk of each product group in the portfolio
#   - Credit = credit_rate × min(scan_risk_leg1, scan_risk_leg2)
#   - This recognises that correlated products partially offset each other
#
# Note: inter credit is applied AFTER intra credits reduce scan risk
# So we work on residual scan risk (post-intra) per product
# =============================================================================

def compute_intercommodity_credits(
    scan_risk_summary: pd.DataFrame,
    intra_credits: dict
) -> list:
    """
    Computes inter-commodity spread credits for defined product pairs.
    Works on residual scan risk after intra-commodity credits.
    """
    # Build residual scan risk per product (post-intra)
    scan_df = scan_risk_summary.copy()
    scan_df["product_code"] = scan_df["Contract"].apply(
        lambda x: CONTRACT_SPECS[x].product_code
    )

    product_scan_risk = scan_df.groupby("product_code")["Scan_Risk ($)"].sum().to_dict()

    # Subtract intra credits from the relevant product's scan risk
    residual_risk = product_scan_risk.copy()
    for product, info in intra_credits.items():
        residual_risk[product] = max(
            0.0,
            residual_risk.get(product, 0) - info["net_intra_credit"]
        )

    inter_results = []
    for spread_def in INTERCOMMODITY_SPREADS:
        p1 = spread_def.leg1_product
        p2 = spread_def.leg2_product

        risk1 = residual_risk.get(p1, 0.0)
        risk2 = residual_risk.get(p2, 0.0)

        # Only credit if both legs are present in portfolio
        if risk1 == 0.0 or risk2 == 0.0:
            continue

        credit = spread_def.credit_rate * min(risk1, risk2)

        inter_results.append({
            "pair"              : f"{p1} / {p2}",
            "leg1_product"      : p1,
            "leg2_product"      : p2,
            "residual_risk_leg1": round(risk1, 2),
            "residual_risk_leg2": round(risk2, 2),
            "credit_rate"       : spread_def.credit_rate,
            "inter_credit"      : round(credit, 2),
        })

    return inter_results


# =============================================================================
# 3.3 TOTAL SPREADING SUMMARY
# =============================================================================

def compute_total_spreading(scan_risk_summary: pd.DataFrame) -> dict:
    intra = compute_intracommodity_credits(scan_risk_summary)
    inter = compute_intercommodity_credits(scan_risk_summary, intra)

    total_intra_credit = sum(v["net_intra_credit"] for v in intra.values())
    total_inter_credit = sum(v["inter_credit"]     for v in inter)
    total_credit       = total_intra_credit + total_inter_credit

    return {
        "intra_credits"      : intra,
        "inter_credits"      : inter,
        "total_intra_credit" : round(total_intra_credit, 2),
        "total_inter_credit" : round(total_inter_credit, 2),
        "total_credit"       : round(total_credit, 2),
    }


# =============================================================================
# 3.4 PRINT OUTPUTS
# =============================================================================

def print_spreading_report(spreading: dict, gross_scan_risk: float):

    print(f"\n{'='*65}")
    print("MODULE 3 — SPREADING CREDITS REPORT")
    print(f"{'='*65}")

    # --- Intra-commodity ---
    print(f"\n{'─'*65}")
    print("  INTRACOMMODITY (CALENDAR) SPREADS")
    print(f"{'─'*65}")
    if not spreading["intra_credits"]:
        print("  No calendar spreads detected.")
    else:
        for product, info in spreading["intra_credits"].items():
            print(f"\n  Product          : {product}")
            print(f"  Long legs        : {info['long_contracts']}")
            print(f"  Short legs       : {info['short_contracts']}")
            print(f"  Num Spreads      : {info['num_spreads']}")
            print(f"  Avg Long Risk/lot: ${info['avg_long_risk_per_lot']:>10,.2f}")
            print(f"  Avg Short Risk/lot:${info['avg_short_risk_per_lot']:>10,.2f}")
            print(f"  Gross Credit     : ${info['gross_credit']:>10,.2f}")
            print(f"  Spread Charge    : ${info['spread_charge']:>10,.2f}  (penalty)")
            print(f"  Net Intra Credit : ${info['net_intra_credit']:>10,.2f}")

    print(f"\n  Total Intra Credit : ${spreading['total_intra_credit']:>10,.2f}")

    # --- Inter-commodity ---
    print(f"\n{'─'*65}")
    print("  INTERCOMMODITY SPREADS")
    print(f"{'─'*65}")
    if not spreading["inter_credits"]:
        print("  No inter-commodity spreads detected.")
    else:
        for info in spreading["inter_credits"]:
            print(f"\n  Pair             : {info['pair']}")
            print(f"  Residual Risk {info['leg1_product']}  : ${info['residual_risk_leg1']:>10,.2f}")
            print(f"  Residual Risk {info['leg2_product']}  : ${info['residual_risk_leg2']:>10,.2f}")
            print(f"  Credit Rate      : {info['credit_rate']*100:.0f}%")
            print(f"  Inter Credit     : ${info['inter_credit']:>10,.2f}")

    print(f"\n  Total Inter Credit : ${spreading['total_inter_credit']:>10,.2f}")

    # --- Summary ---
    print(f"\n{'─'*65}")
    print("  SPREADING SUMMARY")
    print(f"{'─'*65}")
    print(f"  Gross Scan Risk        : ${gross_scan_risk:>10,.2f}")
    print(f"  (-) Intra Credit       : ${spreading['total_intra_credit']:>10,.2f}")
    print(f"  (-) Inter Credit       : ${spreading['total_inter_credit']:>10,.2f}")
    net_after = gross_scan_risk - spreading["total_credit"]
    print(f"  {'─'*40}")
    print(f"  Net Scan Risk (pre-agg): ${net_after:>10,.2f}")
    print(f"  Total Spreading Relief : {spreading['total_credit']/gross_scan_risk*100:.1f}% of gross risk")
    print(f"{'='*65}")


# =============================================================================
# 3.5 RUN
# =============================================================================

if __name__ == "__main__":

    # Pull scan risk summary from Module 2
    scan_summary  = build_scan_risk_summary()
    gross_scan    = scan_summary["Scan_Risk ($)"].sum()

    # Compute spreading
    spreading     = compute_total_spreading(scan_summary)

    # Print
    print_spreading_report(spreading, gross_scan)


MODULE 3 — SPREADING CREDITS REPORT

─────────────────────────────────────────────────────────────────
  INTRACOMMODITY (CALENDAR) SPREADS
─────────────────────────────────────────────────────────────────

  Product          : CL
  Long legs        : ['CL_Mar25']
  Short legs       : ['CL_Jun25']
  Num Spreads      : 3
  Avg Long Risk/lot: $  4,000.00
  Avg Short Risk/lot:$  8,000.00
  Gross Credit     : $ 36,000.00
  Spread Charge    : $    600.00  (penalty)
  Net Intra Credit : $ 35,400.00

  Total Intra Credit : $ 35,400.00

─────────────────────────────────────────────────────────────────
  INTERCOMMODITY SPREADS
─────────────────────────────────────────────────────────────────

  Pair             : CL / NG
  Residual Risk CL  : $  8,600.00
  Residual Risk NG  : $ 20,000.00
  Credit Rate      : 50%
  Inter Credit     : $  4,300.00

  Total Inter Credit : $  4,300.00

─────────────────────────────────────────────────────────────────
  SPREADING SUMMARY
─────────────────────────────

In [4]:
# =============================================================================
# MODULE 4: MARGIN AGGREGATOR
# Allocates spreading credits back to positions, computes final SPAN margin
# per position and at portfolio level
# =============================================================================

# Paste Modules 1, 2, and 3 above this before running

# =============================================================================
# 4.1 ALLOCATE SPREADING CREDITS TO POSITIONS
#
# Logic:
# - Intra credit is split equally across the paired legs (long + short)
#   proportional to each leg's contribution to gross credit
# - Inter credit is allocated to the leg with lower residual risk
#   (the "smaller side" that drove the credit calculation)
# - Remaining positions (unhedged lots) carry their full scan risk
# =============================================================================

def allocate_credits_to_positions(
    scan_risk_summary: pd.DataFrame,
    spreading: dict
) -> pd.DataFrame:
    """
    Returns a DataFrame with per-position credit allocation and net margin.
    """
    df = scan_risk_summary.copy()
    df["product_code"]  = df["Contract"].apply(lambda x: CONTRACT_SPECS[x].product_code)
    df["net_pos"]       = df["Contract"].apply(lambda x: PORTFOLIO[x].net_position)
    df["lots"]          = df["net_pos"].abs()
    df["intra_credit_allocated"] = 0.0
    df["inter_credit_allocated"] = 0.0

    # --- Intra credit allocation ---
    for product, info in spreading["intra_credits"].items():
        num_spreads     = info["num_spreads"]
        long_contracts  = info["long_contracts"]
        short_contracts = info["short_contracts"]

        # Credit each long leg: avg_long_risk_per_lot * num_spreads
        long_mask = df["Contract"].isin(long_contracts)
        long_lots = df.loc[long_mask, "lots"].sum()
        for contract in long_contracts:
            lot_share = df.loc[df["Contract"] == contract, "lots"].values[0] / long_lots
            credit    = info["avg_long_risk_per_lot"] * num_spreads * lot_share
            df.loc[df["Contract"] == contract, "intra_credit_allocated"] += credit

        # Credit each short leg: avg_short_risk_per_lot * num_spreads
        short_mask = df["Contract"].isin(short_contracts)
        short_lots = df.loc[short_mask, "lots"].sum()
        for contract in short_contracts:
            lot_share = df.loc[df["Contract"] == contract, "lots"].values[0] / short_lots
            credit    = info["avg_short_risk_per_lot"] * num_spreads * lot_share
            df.loc[df["Contract"] == contract, "intra_credit_allocated"] += credit

    # --- Inter credit allocation ---
    # Credit goes to the smaller-residual-risk product (the binding leg)
    for info in spreading["inter_credits"]:
        p1, p2      = info["leg1_product"], info["leg2_product"]
        r1, r2      = info["residual_risk_leg1"], info["residual_risk_leg2"]
        credit      = info["inter_credit"]
        # Smaller side gets the credit
        beneficiary = p1 if r1 <= r2 else p2

        # Distribute proportionally across contracts in that product
        bene_mask   = df["product_code"] == beneficiary
        total_risk  = df.loc[bene_mask, "Scan_Risk ($)"].sum()
        for idx in df[bene_mask].index:
            share = df.loc[idx, "Scan_Risk ($)"] / total_risk if total_risk > 0 else 0
            df.loc[idx, "inter_credit_allocated"] += credit * share

    # --- Net margin per position ---
    df["total_credit"]  = df["intra_credit_allocated"] + df["inter_credit_allocated"]
    df["span_margin"]   = (df["Scan_Risk ($)"] - df["total_credit"]).clip(lower=0).round(2)

    df["intra_credit_allocated"] = df["intra_credit_allocated"].round(2)
    df["inter_credit_allocated"] = df["inter_credit_allocated"].round(2)
    df["total_credit"]           = df["total_credit"].round(2)

    return df[[
        "Contract", "net_pos", "Scan_Risk ($)",
        "intra_credit_allocated", "inter_credit_allocated",
        "total_credit", "span_margin"
    ]].rename(columns={
        "net_pos"                  : "Net_Pos",
        "intra_credit_allocated"   : "Intra_Credit ($)",
        "inter_credit_allocated"   : "Inter_Credit ($)",
        "total_credit"             : "Total_Credit ($)",
        "span_margin"              : "SPAN_Margin ($)",
    })


# =============================================================================
# 4.2 PORTFOLIO-LEVEL SPAN MARGIN SUMMARY
# =============================================================================

def compute_portfolio_margin(margin_df: pd.DataFrame, spreading: dict) -> dict:

    gross_scan      = margin_df["Scan_Risk ($)"].sum()
    total_intra     = spreading["total_intra_credit"]
    total_inter     = spreading["total_inter_credit"]
    total_credit    = spreading["total_credit"]
    net_margin      = margin_df["SPAN_Margin ($)"].sum()

    return {
        "gross_scan_risk"       : round(gross_scan, 2),
        "intra_credit"          : round(total_intra, 2),
        "inter_credit"          : round(total_inter, 2),
        "total_credit"          : round(total_credit, 2),
        "portfolio_span_margin" : round(net_margin, 2),
        "margin_efficiency"     : round((1 - net_margin / gross_scan) * 100, 1),
    }


# =============================================================================
# 4.3 PRINT OUTPUTS
# =============================================================================

def print_margin_per_position(margin_df: pd.DataFrame):
    print(f"\n{'='*85}")
    print("MODULE 4 — PER-POSITION SPAN MARGIN")
    print(f"{'='*85}")
    print(f"\n{'Contract':<12} {'Net_Pos':>8} {'ScanRisk':>12} {'IntraCredit':>13} "
          f"{'InterCredit':>13} {'TotalCredit':>13} {'SPANMargin':>12}")
    print("-" * 85)
    for _, row in margin_df.iterrows():
        print(f"{row['Contract']:<12} {int(row['Net_Pos']):>8} "
              f"{row['Scan_Risk ($)']:>12,.2f} "
              f"{row['Intra_Credit ($)']:>13,.2f} "
              f"{row['Inter_Credit ($)']:>13,.2f} "
              f"{row['Total_Credit ($)']:>13,.2f} "
              f"{row['SPAN_Margin ($)']:>12,.2f}")
    print("-" * 85)
    print(f"{'TOTAL':<12} {'':>8} "
          f"{margin_df['Scan_Risk ($)'].sum():>12,.2f} "
          f"{margin_df['Intra_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Inter_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Total_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['SPAN_Margin ($)'].sum():>12,.2f}")
    print(f"{'='*85}")


def print_portfolio_margin_summary(summary: dict):
    print(f"\n{'='*55}")
    print("  PORTFOLIO SPAN MARGIN SUMMARY")
    print(f"{'='*55}")
    print(f"  Gross Scan Risk             : ${summary['gross_scan_risk']:>12,.2f}")
    print(f"  (-) Intra-Commodity Credit  : ${summary['intra_credit']:>12,.2f}")
    print(f"  (-) Inter-Commodity Credit  : ${summary['inter_credit']:>12,.2f}")
    print(f"  {'─'*42}")
    print(f"  PORTFOLIO SPAN MARGIN       : ${summary['portfolio_span_margin']:>12,.2f}")
    print(f"  {'─'*42}")
    print(f"  Margin Efficiency           : {summary['margin_efficiency']:>11.1f}%")
    print(f"  (% gross risk offset by spreading)")
    print(f"{'='*55}")


# =============================================================================
# 4.4 RUN
# =============================================================================

if __name__ == "__main__":

    # --- Pull inputs from Modules 2 and 3 ---
    scan_summary  = build_scan_risk_summary()
    gross_scan    = scan_summary["Scan_Risk ($)"].sum()
    spreading     = compute_total_spreading(scan_summary)

    # --- Allocate credits and compute per-position margin ---
    margin_df     = allocate_credits_to_positions(scan_summary, spreading)

    # --- Portfolio-level summary ---
    port_summary  = compute_portfolio_margin(margin_df, spreading)

    # --- Print ---
    print_margin_per_position(margin_df)
    print_portfolio_margin_summary(port_summary)


MODULE 4 — PER-POSITION SPAN MARGIN

Contract      Net_Pos     ScanRisk   IntraCredit   InterCredit   TotalCredit   SPANMargin
-------------------------------------------------------------------------------------
CL_Mar25            5    20,000.00     12,000.00      1,954.55     13,954.55     6,045.45
CL_Jun25           -3    24,000.00     24,000.00      2,345.45     26,345.45         0.00
NG_Mar25            4    20,000.00          0.00          0.00          0.00    20,000.00
GC_Apr25            2     7,000.00          0.00          0.00          0.00     7,000.00
-------------------------------------------------------------------------------------
TOTAL                    71,000.00     36,000.00      4,300.00     40,300.00    33,045.45

  PORTFOLIO SPAN MARGIN SUMMARY
  Gross Scan Risk             : $   71,000.00
  (-) Intra-Commodity Credit  : $   35,400.00
  (-) Inter-Commodity Credit  : $    4,300.00
  ──────────────────────────────────────────
  PORTFOLIO SPAN MARGIN       : $ 

In [5]:
# =============================================================================
# MODULE 4 (REVISED v2): MARGIN AGGREGATOR — spread charge fix
# =============================================================================

def allocate_credits_to_positions(
    scan_risk_summary: pd.DataFrame,
    spreading: dict
) -> pd.DataFrame:

    df = scan_risk_summary.copy()
    df["product_code"]           = df["Contract"].apply(lambda x: CONTRACT_SPECS[x].product_code)
    df["net_pos"]                = df["Contract"].apply(lambda x: PORTFOLIO[x].net_position)
    df["lots"]                   = df["net_pos"].abs()
    df["intra_credit_allocated"] = 0.0
    df["inter_credit_allocated"] = 0.0

    # --- Intra credit ---
    # Allocate NET intra credit (after spread charge) proportionally
    # across all legs involved in the spread, weighted by gross credit contribution
    for product, info in spreading["intra_credits"].items():
        num_spreads     = info["num_spreads"]
        long_contracts  = info["long_contracts"]
        short_contracts = info["short_contracts"]
        net_credit      = info["net_intra_credit"]   # KEY FIX: use net not gross

        # Gross credit contribution per leg
        long_lots  = sum(df.loc[df["Contract"] == c, "lots"].values[0] for c in long_contracts)
        short_lots = sum(df.loc[df["Contract"] == c, "lots"].values[0] for c in short_contracts)

        gross_long_credit  = info["avg_long_risk_per_lot"]  * num_spreads
        gross_short_credit = info["avg_short_risk_per_lot"] * num_spreads
        gross_total        = gross_long_credit + gross_short_credit

        # Distribute net credit proportionally by gross contribution weight
        long_share  = gross_long_credit  / gross_total
        short_share = gross_short_credit / gross_total

        for contract in long_contracts:
            lot_share = df.loc[df["Contract"] == contract, "lots"].values[0] / long_lots
            df.loc[df["Contract"] == contract, "intra_credit_allocated"] += (
                net_credit * long_share * lot_share
            )

        for contract in short_contracts:
            lot_share = df.loc[df["Contract"] == contract, "lots"].values[0] / short_lots
            df.loc[df["Contract"] == contract, "intra_credit_allocated"] += (
                net_credit * short_share * lot_share
            )

    # --- Inter credit ---
    for info in spreading["inter_credits"]:
        p1, p2      = info["leg1_product"], info["leg2_product"]
        r1, r2      = info["residual_risk_leg1"], info["residual_risk_leg2"]
        credit      = info["inter_credit"]
        beneficiary = p1 if r1 <= r2 else p2

        bene_mask  = df["product_code"] == beneficiary
        total_risk = df.loc[bene_mask, "Scan_Risk ($)"].sum()
        for idx in df[bene_mask].index:
            share = df.loc[idx, "Scan_Risk ($)"] / total_risk if total_risk > 0 else 0
            df.loc[idx, "inter_credit_allocated"] += credit * share

    df["total_credit"] = df["intra_credit_allocated"] + df["inter_credit_allocated"]
    return df


def reallocate_excess_credits(df: pd.DataFrame, max_iter: int = 10) -> pd.DataFrame:
    working = df.copy()
    working["total_credit"] = working["intra_credit_allocated"] + working["inter_credit_allocated"]

    for iteration in range(max_iter):
        working["raw_margin"] = working["Scan_Risk ($)"] - working["total_credit"]
        working["excess"]     = (-working["raw_margin"]).clip(lower=0)
        working["absorbable"] = working["raw_margin"].clip(lower=0)

        total_excess     = working["excess"].sum()
        total_absorbable = working["absorbable"].sum()

        if total_excess < 0.01:
            break

        if total_absorbable < 0.01:
            break

        for idx in working.index:
            if working.loc[idx, "absorbable"] > 0:
                share = working.loc[idx, "absorbable"] / total_absorbable
                working.loc[idx, "total_credit"] += total_excess * share

        for idx in working.index:
            if working.loc[idx, "total_credit"] > working.loc[idx, "Scan_Risk ($)"]:
                working.loc[idx, "total_credit"] = working.loc[idx, "Scan_Risk ($)"]

    working["span_margin"]  = (working["Scan_Risk ($)"] - working["total_credit"]).clip(lower=0).round(2)
    working["total_credit"] = working["total_credit"].round(2)
    return working


def compute_span_margins(scan_risk_summary, spreading):
    df = allocate_credits_to_positions(scan_risk_summary, spreading)
    df = reallocate_excess_credits(df)
    return df[[
        "Contract", "net_pos", "Scan_Risk ($)",
        "intra_credit_allocated", "inter_credit_allocated",
        "total_credit", "span_margin"
    ]].rename(columns={
        "net_pos"                : "Net_Pos",
        "intra_credit_allocated" : "Intra_Credit ($)",
        "inter_credit_allocated" : "Inter_Credit ($)",
        "total_credit"           : "Total_Credit ($)",
        "span_margin"            : "SPAN_Margin ($)",
    })


def compute_portfolio_margin(margin_df, spreading):
    gross_scan   = margin_df["Scan_Risk ($)"].sum()
    net_margin   = margin_df["SPAN_Margin ($)"].sum()
    total_credit = gross_scan - net_margin
    return {
        "gross_scan_risk"       : round(gross_scan, 2),
        "intra_credit"          : round(spreading["total_intra_credit"], 2),
        "inter_credit"          : round(spreading["total_inter_credit"], 2),
        "total_credit"          : round(total_credit, 2),
        "portfolio_span_margin" : round(net_margin, 2),
        "margin_efficiency"     : round((total_credit / gross_scan) * 100, 1),
    }


def print_margin_per_position(margin_df):
    print(f"\n{'='*85}")
    print("MODULE 4 (REVISED v2) — PER-POSITION SPAN MARGIN  [spread charge fixed]")
    print(f"{'='*85}")
    print(f"\n{'Contract':<12} {'Net_Pos':>8} {'ScanRisk':>12} {'IntraCredit':>13} "
          f"{'InterCredit':>13} {'TotalCredit':>13} {'SPANMargin':>12}")
    print("-" * 85)
    for _, row in margin_df.iterrows():
        print(f"{row['Contract']:<12} {int(row['Net_Pos']):>8} "
              f"{row['Scan_Risk ($)']:>12,.2f} "
              f"{row['Intra_Credit ($)']:>13,.2f} "
              f"{row['Inter_Credit ($)']:>13,.2f} "
              f"{row['Total_Credit ($)']:>13,.2f} "
              f"{row['SPAN_Margin ($)']:>12,.2f}")
    print("-" * 85)
    print(f"{'TOTAL':<12} {'':>8} "
          f"{margin_df['Scan_Risk ($)'].sum():>12,.2f} "
          f"{margin_df['Intra_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Inter_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Total_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['SPAN_Margin ($)'].sum():>12,.2f}")
    print(f"{'='*85}")


def print_portfolio_margin_summary(summary):
    print(f"\n{'='*55}")
    print("  PORTFOLIO SPAN MARGIN SUMMARY")
    print(f"{'='*55}")
    print(f"  Gross Scan Risk             : ${summary['gross_scan_risk']:>12,.2f}")
    print(f"  (-) Intra-Commodity Credit  : ${summary['intra_credit']:>12,.2f}")
    print(f"  (-) Inter-Commodity Credit  : ${summary['inter_credit']:>12,.2f}")
    print(f"  {'─'*42}")
    print(f"  PORTFOLIO SPAN MARGIN       : ${summary['portfolio_span_margin']:>12,.2f}")
    print(f"  {'─'*42}")
    print(f"  Margin Efficiency           : {summary['margin_efficiency']:>11.1f}%")
    print(f"  (% gross risk offset by spreading)")
    print(f"{'='*55}")


if __name__ == "__main__":
    scan_summary = build_scan_risk_summary()
    spreading    = compute_total_spreading(scan_summary)
    margin_df    = compute_span_margins(scan_summary, spreading)
    port_summary = compute_portfolio_margin(margin_df, spreading)
    print_margin_per_position(margin_df)
    print_portfolio_margin_summary(port_summary)


MODULE 4 (REVISED v2) — PER-POSITION SPAN MARGIN  [spread charge fixed]

Contract      Net_Pos     ScanRisk   IntraCredit   InterCredit   TotalCredit   SPANMargin
-------------------------------------------------------------------------------------
CL_Mar25            5    20,000.00     11,800.00      1,954.55     14,120.02     5,879.98
CL_Jun25           -3    24,000.00     23,600.00      2,345.45     24,000.00         0.00
NG_Mar25            4    20,000.00          0.00          0.00      1,170.36    18,829.64
GC_Apr25            2     7,000.00          0.00          0.00        409.63     6,590.37
-------------------------------------------------------------------------------------
TOTAL                    71,000.00     35,400.00      4,300.00     39,700.01    31,299.99

  PORTFOLIO SPAN MARGIN SUMMARY
  Gross Scan Risk             : $   71,000.00
  (-) Intra-Commodity Credit  : $   35,400.00
  (-) Inter-Commodity Credit  : $    4,300.00
  ─────────────────────────────────────────

In [6]:
# =============================================================================
# MODULE 5: OUTPUT LAYER + MASTER RUNNER
# Full SPAN report — scenario summary, margin waterfall, audit trail
# Single entry point: run_span_model()
# =============================================================================

# Paste Modules 1, 2, 3, 4 (revised v2) above this before running

import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# 5.1 SCENARIO P&L SUMMARY TABLE
# Condensed view — worst scenario per contract highlighted
# =============================================================================

def build_scenario_summary() -> pd.DataFrame:
    rows = []
    for cid in PORTFOLIO:
        df      = compute_scenario_pnl(cid)
        pos     = PORTFOLIO[cid]
        spec    = CONTRACT_SPECS[cid]
        params  = SPAN_PARAMS[spec.product_code]

        worst_idx   = df["Weighted_PnL"].idxmin()
        best_idx    = df["Weighted_PnL"].idxmax()

        rows.append({
            "Contract"          : cid,
            "Direction"         : "LONG" if pos.net_position > 0 else "SHORT",
            "Lots"              : abs(pos.net_position),
            "Settlement"        : pos.current_price,
            "Scan_Range"        : params.price_scan_range,
            "Best_Scenario"     : int(df.loc[best_idx,  "Scenario"]),
            "Best_PnL ($)"      : df.loc[best_idx,  "Weighted_PnL"],
            "Worst_Scenario"    : int(df.loc[worst_idx, "Scenario"]),
            "Worst_PnL ($)"     : df.loc[worst_idx, "Weighted_PnL"],
            "Scan_Risk ($)"     : max(0, -df.loc[worst_idx, "Weighted_PnL"]),
        })
    return pd.DataFrame(rows)


# =============================================================================
# 5.2 MARGIN WATERFALL
# Step-by-step reduction from gross scan risk to final SPAN margin
# =============================================================================

def build_margin_waterfall(spreading: dict, portfolio_summary: dict) -> pd.DataFrame:
    steps = [
        ("Gross Scan Risk",            portfolio_summary["gross_scan_risk"],    ""),
        ("(-) Intra-Commodity Credit", -spreading["total_intra_credit"],        "Calendar spread offset"),
        ("(-) Inter-Commodity Credit", -spreading["total_inter_credit"],        "CL/NG correlation credit"),
        ("= Portfolio SPAN Margin",    portfolio_summary["portfolio_span_margin"], "Final margin requirement"),
    ]
    rows = []
    running = 0.0
    for label, amount, note in steps:
        running += amount
        rows.append({
            "Step"      : label,
            "Amount ($)": amount,
            "Running ($)": running if "=" in label or label == "Gross Scan Risk" else None,
            "Note"      : note,
        })
    # Fix first row running total
    rows[0]["Running ($)"] = portfolio_summary["gross_scan_risk"]
    return pd.DataFrame(rows)


# =============================================================================
# 5.3 PRINT FUNCTIONS
# =============================================================================

def print_header(title: str, width: int = 75):
    print(f"\n{'═'*width}")
    pad = (width - len(title) - 2) // 2
    print(f"{'═'*pad} {title} {'═'*(width - pad - len(title) - 2)}")
    print(f"{'═'*width}")


def print_run_metadata():
    print_header("CME SPAN DUMMY MODEL — FULL REPORT")
    print(f"\n  Run Timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Model Version  : SPAN (Futures + Calendar/Inter-Commodity Spreads)")
    print(f"  Positions      : {len(PORTFOLIO)} contracts across "
          f"{len(set(CONTRACT_SPECS[c].product_code for c in PORTFOLIO))} products")
    print(f"  Scenario Grid  : 16 scenarios (±1/3, ±2/3, ±1x, ±2x scan range + extremes)")


def print_position_summary():
    print(f"\n{'─'*75}")
    print("  SECTION 1 — PORTFOLIO POSITIONS")
    print(f"{'─'*75}")
    print(f"\n  {'Contract':<12} {'Dir':<6} {'Lots':>5} {'Entry':>10} "
          f"{'Settlement':>12} {'PointVal':>10} {'Group':<10}")
    print(f"  {'-'*70}")
    for cid, pos in PORTFOLIO.items():
        spec = CONTRACT_SPECS[cid]
        direction = "LONG" if pos.net_position > 0 else "SHORT"
        print(f"  {cid:<12} {direction:<6} {abs(pos.net_position):>5} "
              f"{pos.entry_price:>10.3f} {pos.current_price:>12.3f} "
              f"{spec.point_value:>10.0f} {spec.product_group:<10}")


def print_scenario_summary(scenario_df: pd.DataFrame):
    print(f"\n{'─'*75}")
    print("  SECTION 2 — SCENARIO ANALYSIS SUMMARY")
    print(f"{'─'*75}")
    print(f"\n  {'Contract':<12} {'Dir':<6} {'Lots':>5} {'ScanRange':>10} "
          f"{'BestScen':>9} {'BestPnL':>12} {'WorstScen':>10} {'WorstPnL':>12} {'ScanRisk':>11}")
    print(f"  {'-'*70}")
    for _, row in scenario_df.iterrows():
        print(f"  {row['Contract']:<12} {row['Direction']:<6} {int(row['Lots']):>5} "
              f"{row['Scan_Range']:>10.3f} "
              f"{'S'+str(row['Best_Scenario']):>9} {row['Best_PnL ($)']:>12,.2f} "
              f"{'S'+str(row['Worst_Scenario']):>10} {row['Worst_PnL ($)']:>12,.2f} "
              f"{row['Scan_Risk ($)']:>11,.2f}")
    print(f"  {'-'*70}")
    print(f"  {'TOTAL GROSS SCAN RISK':>60} : ${scenario_df['Scan_Risk ($)'].sum():>10,.2f}")


def print_spreading_summary(spreading: dict):
    print(f"\n{'─'*75}")
    print("  SECTION 3 — SPREADING CREDITS")
    print(f"{'─'*75}")

    print(f"\n  INTRACOMMODITY SPREADS")
    print(f"  {'─'*55}")
    if spreading["intra_credits"]:
        for product, info in spreading["intra_credits"].items():
            print(f"  Product       : {product}")
            print(f"  Legs          : {info['long_contracts']} (long)  ×  "
                  f"{info['short_contracts']} (short)")
            print(f"  Pairs         : {info['num_spreads']} spreads")
            print(f"  Gross Credit  : ${info['gross_credit']:>10,.2f}")
            print(f"  Spread Charge : ${info['spread_charge']:>10,.2f}  (penalty @ "
                  f"${SPAN_PARAMS[product].intracommodity_spread_charge:.0f}/spread)")
            print(f"  Net Credit    : ${info['net_intra_credit']:>10,.2f}")
    print(f"\n  Total Intra Credit : ${spreading['total_intra_credit']:>10,.2f}")

    print(f"\n  INTERCOMMODITY SPREADS")
    print(f"  {'─'*55}")
    if spreading["inter_credits"]:
        for info in spreading["inter_credits"]:
            print(f"  Pair          : {info['pair']}")
            print(f"  Residual Risk : {info['leg1_product']} ${info['residual_risk_leg1']:,.2f}  |  "
                  f"{info['leg2_product']} ${info['residual_risk_leg2']:,.2f}")
            print(f"  Credit Rate   : {info['credit_rate']*100:.0f}%  ×  "
                  f"min(${info['residual_risk_leg1']:,.2f}, ${info['residual_risk_leg2']:,.2f})")
            print(f"  Inter Credit  : ${info['inter_credit']:>10,.2f}")
    print(f"\n  Total Inter Credit : ${spreading['total_inter_credit']:>10,.2f}")


def print_per_position_margin(margin_df: pd.DataFrame):
    print(f"\n{'─'*75}")
    print("  SECTION 4 — PER-POSITION SPAN MARGIN")
    print(f"{'─'*75}")
    print(f"\n  {'Contract':<12} {'Net_Pos':>8} {'ScanRisk':>12} "
          f"{'IntraCredit':>13} {'InterCredit':>13} {'TotalCredit':>13} {'SPANMargin':>12}")
    print(f"  {'-'*83}")
    for _, row in margin_df.iterrows():
        print(f"  {row['Contract']:<12} {int(row['Net_Pos']):>8} "
              f"{row['Scan_Risk ($)']:>12,.2f} "
              f"{row['Intra_Credit ($)']:>13,.2f} "
              f"{row['Inter_Credit ($)']:>13,.2f} "
              f"{row['Total_Credit ($)']:>13,.2f} "
              f"{row['SPAN_Margin ($)']:>12,.2f}")
    print(f"  {'-'*83}")
    print(f"  {'TOTAL':<12} {'':>8} "
          f"{margin_df['Scan_Risk ($)'].sum():>12,.2f} "
          f"{margin_df['Intra_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Inter_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['Total_Credit ($)'].sum():>13,.2f} "
          f"{margin_df['SPAN_Margin ($)'].sum():>12,.2f}")


def print_margin_waterfall(waterfall_df: pd.DataFrame, portfolio_summary: dict):
    print(f"\n{'─'*75}")
    print("  SECTION 5 — MARGIN WATERFALL")
    print(f"{'─'*75}\n")
    for _, row in waterfall_df.iterrows():
        amount_str  = f"${abs(row['Amount ($)']):>12,.2f}"
        sign        = "  " if row["Amount ($)"] >= 0 else "- "
        running_str = f"  →  Running: ${row['Running ($)']:>12,.2f}" \
                      if pd.notna(row["Running ($)"]) else ""
        note_str    = f"  [{row['Note']}]" if row["Note"] else ""
        print(f"  {row['Step']:<35} {sign}{amount_str}{running_str}{note_str}")

    print(f"\n{'─'*75}")
    print(f"  {'PORTFOLIO SPAN MARGIN':<35}    ${portfolio_summary['portfolio_span_margin']:>12,.2f}")
    print(f"  {'Margin Efficiency':<35}    {portfolio_summary['margin_efficiency']:>11.1f}%")
    print(f"  {'Gross-to-Net Relief':<35}    "
          f"${portfolio_summary['gross_scan_risk'] - portfolio_summary['portfolio_span_margin']:>12,.2f}")


def print_footer():
    print(f"\n{'═'*75}")
    print("  MODEL ASSUMPTIONS & LIMITATIONS")
    print(f"{'─'*75}")
    notes = [
        "Futures only — options/delta-adjusted positions not modelled",
        "Vol shift has no P&L impact on futures (included for completeness)",
        "Extreme scenarios (S15/S16) weighted at 35% per CME standard",
        "Intra spread charge applied as flat penalty per spread pair",
        "Inter credit applied to smaller residual risk leg only",
        "Floating point residual < $0.01 from iterative reallocation loop",
    ]
    for i, note in enumerate(notes, 1):
        print(f"  {i}. {note}")
    print(f"{'═'*75}\n")


# =============================================================================
# 5.4 MASTER RUNNER — single entry point
# =============================================================================

def run_span_model():
    """
    Full SPAN model pipeline.
    Runs all 5 modules and prints the complete report.
    Returns all computed objects for programmatic access.
    """
    # --- Module 2 ---
    scan_summary    = build_scan_risk_summary()

    # --- Module 3 ---
    spreading       = compute_total_spreading(scan_summary)

    # --- Module 4 ---
    margin_df       = compute_span_margins(scan_summary, spreading)
    port_summary    = compute_portfolio_margin(margin_df, spreading)

    # --- Module 5 outputs ---
    scenario_df     = build_scenario_summary()
    waterfall_df    = build_margin_waterfall(spreading, port_summary)

    # --- Print full report ---
    print_run_metadata()
    print_position_summary()
    print_scenario_summary(scenario_df)
    print_spreading_summary(spreading)
    print_per_position_margin(margin_df)
    print_margin_waterfall(waterfall_df, port_summary)
    print_footer()

    return {
        "scan_summary"   : scan_summary,
        "spreading"      : spreading,
        "margin_df"      : margin_df,
        "port_summary"   : port_summary,
        "scenario_df"    : scenario_df,
        "waterfall_df"   : waterfall_df,
    }


# =============================================================================
# 5.5 RUN
# =============================================================================

if __name__ == "__main__":
    results = run_span_model()


═══════════════════════════════════════════════════════════════════════════
═══════════════════ CME SPAN DUMMY MODEL — FULL REPORT ════════════════════
═══════════════════════════════════════════════════════════════════════════

  Run Timestamp  : 2026-03-19 08:29:01
  Model Version  : SPAN (Futures + Calendar/Inter-Commodity Spreads)
  Positions      : 4 contracts across 3 products
  Scenario Grid  : 16 scenarios (±1/3, ±2/3, ±1x, ±2x scan range + extremes)

───────────────────────────────────────────────────────────────────────────
  SECTION 1 — PORTFOLIO POSITIONS
───────────────────────────────────────────────────────────────────────────

  Contract     Dir     Lots      Entry   Settlement   PointVal Group     
  ----------------------------------------------------------------------
  CL_Mar25     LONG       5     78.500       79.200       1000 ENERGY    
  CL_Jun25     SHORT      3     79.800       80.100       1000 ENERGY    
  NG_Mar25     LONG       4      2.650        2.710  

### **6.1 — Black-76 pricer + Greeks**

In [8]:
# =============================================================================
# MODULE 6: OPTIONS SUPPORT
# Black-76 pricer, scenario P&L, Short Option Minimum, full integration
# =============================================================================

# Paste Modules 1–5 above this before running

import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
from dataclasses import dataclass
from typing import Optional

# =============================================================================
# 6.1 OPTIONS CONTRACT DEFINITIONS
# =============================================================================

@dataclass
class OptionSpec:
    contract_id:     str
    underlying_id:   str        # must match a ContractSpec key
    option_type:     str        # 'call' or 'put'
    strike:          float
    expiry_years:    float      # time to expiry in years
    implied_vol:     float      # annualised IV as decimal (e.g. 0.28)
    point_value:     float      # same as underlying (inherited for clarity)
    product_code:    str
    product_group:   str

OPTION_SPECS: dict = {
    "CL_Mar25_C82": OptionSpec(
        contract_id   = "CL_Mar25_C82",
        underlying_id = "CL_Mar25",
        option_type   = "call",
        strike        = 82.00,
        expiry_years  = 30 / 365,
        implied_vol   = 0.28,
        point_value   = 1000.0,
        product_code  = "CL",
        product_group = "ENERGY",
    ),
    "CL_Mar25_P76": OptionSpec(
        contract_id   = "CL_Mar25_P76",
        underlying_id = "CL_Mar25",
        option_type   = "put",
        strike        = 76.00,
        expiry_years  = 30 / 365,
        implied_vol   = 0.26,
        point_value   = 1000.0,
        product_code  = "CL",
        product_group = "ENERGY",
    ),
    "GC_Apr25_C2100": OptionSpec(
        contract_id   = "GC_Apr25_C2100",
        underlying_id = "GC_Apr25",
        option_type   = "call",
        strike        = 2100.0,
        expiry_years  = 45 / 365,
        implied_vol   = 0.18,
        point_value   = 100.0,
        product_code  = "GC",
        product_group = "METALS",
    ),
}

# Option positions — signed lots (+long, -short)
OPTION_PORTFOLIO: dict = {
    "CL_Mar25_C82"  : {"net_position": -3, "entry_premium": None},   # short 3 — SOM candidate
    "CL_Mar25_P76"  : {"net_position": +2, "entry_premium": None},   # long  2
    "GC_Apr25_C2100": {"net_position": -1, "entry_premium": None},   # short 1
}

# Vol shift magnitude per scenario (CME uses ~30% of IV as vol shift)
# We parameterise it per product
VOL_SHIFT_PARAMS: dict = {
    "CL" : 0.05,    # ±5 vol points absolute shift
    "NG" : 0.07,
    "GC" : 0.04,
}

# Short Option Minimum rate (% of scan range × point value per lot)
# CME sets this so deep OTM shorts always carry some minimum margin
SOM_RATE: dict = {
    "CL": 0.50,    # 50% of scan range × point value as SOM floor per lot
    "NG": 0.50,
    "GC": 0.50,
}

# =============================================================================
# 6.2 BLACK-76 PRICER
# Black-76 is the standard model for options on futures
# F = futures price, K = strike, r = risk-free rate (0 for margining),
# T = time to expiry, sigma = implied vol
# =============================================================================

def black76_price(
    F: float, K: float, T: float, sigma: float,
    option_type: str, r: float = 0.0
) -> float:
    """
    Black-76 option price on a futures contract.
    Returns option premium in price units (per unit of underlying).
    """
    if T <= 0:
        # At expiry: intrinsic value only
        if option_type == "call":
            return max(F - K, 0.0)
        else:
            return max(K - F, 0.0)

    sqrt_T  = np.sqrt(T)
    d1      = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * sqrt_T)
    d2      = d1 - sigma * sqrt_T
    disc    = np.exp(-r * T)

    if option_type == "call":
        return disc * (F * norm.cdf(d1) - K * norm.cdf(d2))
    else:
        return disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1))


def black76_greeks(
    F: float, K: float, T: float, sigma: float,
    option_type: str, r: float = 0.0
) -> dict:
    """
    Returns delta, gamma, vega, theta for a Black-76 option.
    Delta: per unit futures price move
    Vega:  per 1% (0.01) move in IV
    Theta: per calendar day
    """
    if T <= 1e-6:
        return {"delta": 0.0, "gamma": 0.0, "vega": 0.0, "theta": 0.0}

    sqrt_T  = np.sqrt(T)
    d1      = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * sqrt_T)
    d2      = d1 - sigma * sqrt_T
    disc    = np.exp(-r * T)
    nd1     = norm.pdf(d1)

    gamma   = disc * nd1 / (F * sigma * sqrt_T)
    vega    = disc * F * nd1 * sqrt_T * 0.01   # per 1 vol point

    if option_type == "call":
        delta = disc * norm.cdf(d1)
        theta = disc * (
            - (F * nd1 * sigma) / (2 * sqrt_T)
            - r * K * norm.cdf(d2)
            + r * F * norm.cdf(d1)
        ) / 365
    else:
        delta = disc * (norm.cdf(d1) - 1)
        theta = disc * (
            - (F * nd1 * sigma) / (2 * sqrt_T)
            + r * K * norm.cdf(-d2)
            - r * F * norm.cdf(-d1)
        ) / 365

    return {
        "delta" : round(delta, 6),
        "gamma" : round(gamma, 6),
        "vega"  : round(vega,  6),
        "theta" : round(theta, 6),
    }


# =============================================================================
# 6.3 OPTION SCENARIO P&L
# Reprice option under each of 16 scenarios
# P&L = (scenario_premium - current_premium) × point_value × net_position
# Vol shift applied per scenario vol_shift direction
# =============================================================================

def compute_option_scenario_pnl(option_id: str) -> pd.DataFrame:
    """
    Returns 16-scenario P&L table for a single option position.
    """
    spec        = OPTION_SPECS[option_id]
    pos         = OPTION_PORTFOLIO[option_id]
    params      = SPAN_PARAMS[spec.product_code]
    und_price   = PORTFOLIO[spec.underlying_id].current_price
    vol_shift   = VOL_SHIFT_PARAMS.get(spec.product_code, 0.04)

    # Current option premium
    current_premium = black76_price(
        F=und_price, K=spec.strike, T=spec.expiry_years,
        sigma=spec.implied_vol, option_type=spec.option_type
    )

    rows = []
    for i, (price_frac, vol_dir, weight) in enumerate(SCENARIO_GRID, start=1):
        price_move      = price_frac * params.price_scan_range
        hypo_F          = und_price + price_move
        hypo_vol        = spec.implied_vol + (vol_shift if vol_dir == "up" else -vol_shift)
        hypo_vol        = max(hypo_vol, 0.001)   # floor vol at near-zero

        scenario_premium = black76_price(
            F=hypo_F, K=spec.strike, T=spec.expiry_years,
            sigma=hypo_vol, option_type=spec.option_type
        )

        raw_pnl      = (scenario_premium - current_premium) * spec.point_value * pos["net_position"]
        weighted_pnl = raw_pnl * weight

        rows.append({
            "Scenario"        : i,
            "Price_Frac"      : price_frac,
            "Vol_Shift"       : vol_dir,
            "Weight"          : weight,
            "Price_Move"      : round(price_move, 4),
            "Hypo_Futures"    : round(hypo_F, 4),
            "Hypo_Vol"        : round(hypo_vol, 4),
            "Current_Premium" : round(current_premium, 4),
            "Scenario_Premium": round(scenario_premium, 4),
            "Raw_PnL"         : round(raw_pnl, 2),
            "Weighted_PnL"    : round(weighted_pnl, 2),
        })

    return pd.DataFrame(rows)


# =============================================================================
# 6.4 SHORT OPTION MINIMUM (SOM)
# For short options, SPAN imposes a floor margin regardless of scenario scan
# SOM = SOM_rate × price_scan_range × point_value × |lots|
# Final margin = max(scan_risk, SOM)
# =============================================================================

def compute_som(option_id: str) -> float:
    """
    Computes Short Option Minimum for a short option position.
    Returns 0 for long positions (SOM does not apply).
    """
    spec = OPTION_SPECS[option_id]
    pos  = OPTION_PORTFOLIO[option_id]

    if pos["net_position"] >= 0:
        return 0.0    # SOM only applies to short positions

    params  = SPAN_PARAMS[spec.product_code]
    rate    = SOM_RATE.get(spec.product_code, 0.50)
    lots    = abs(pos["net_position"])

    som = rate * params.price_scan_range * spec.point_value * lots
    return round(som, 2)


# =============================================================================
# 6.5 OPTION SCAN RISK (max loss across 16 scenarios, vs SOM floor)
# =============================================================================

def compute_option_scan_risk(option_id: str) -> dict:
    df          = compute_option_scenario_pnl(option_id)
    spec        = OPTION_SPECS[option_id]
    pos         = OPTION_PORTFOLIO[option_id]
    und_price   = PORTFOLIO[spec.underlying_id].current_price

    worst_idx   = df["Weighted_PnL"].idxmin()
    worst_row   = df.loc[worst_idx]
    scan_risk   = max(0.0, -worst_row["Weighted_PnL"])

    som         = compute_som(option_id)
    final_risk  = max(scan_risk, som)   # SOM floor applied here

    greeks      = black76_greeks(
        F=und_price, K=spec.strike, T=spec.expiry_years,
        sigma=spec.implied_vol, option_type=spec.option_type
    )

    current_premium = black76_price(
        F=und_price, K=spec.strike, T=spec.expiry_years,
        sigma=spec.implied_vol, option_type=spec.option_type
    )

    moneyness = "ATM"
    if spec.option_type == "call":
        if und_price < spec.strike * 0.98:  moneyness = "OTM"
        elif und_price > spec.strike * 1.02: moneyness = "ITM"
    else:
        if und_price > spec.strike * 1.02:  moneyness = "OTM"
        elif und_price < spec.strike * 0.98: moneyness = "ITM"

    return {
        "option_id"       : option_id,
        "type"            : spec.option_type.upper(),
        "strike"          : spec.strike,
        "moneyness"       : moneyness,
        "net_position"    : pos["net_position"],
        "current_premium" : round(current_premium, 4),
        "delta"           : greeks["delta"],
        "gamma"           : greeks["gamma"],
        "vega"            : greeks["vega"],
        "scan_risk"       : round(scan_risk, 2),
        "som"             : som,
        "som_binding"     : som > scan_risk,
        "final_risk"      : round(final_risk, 2),
        "worst_scenario"  : int(worst_row["Scenario"]),
        "worst_pnl"       : worst_row["Weighted_PnL"],
    }


# =============================================================================
# 6.6 COMBINED SCAN RISK (futures + options)
# Merge option scan risks into the existing scan_summary DataFrame
# =============================================================================

def build_combined_scan_risk() -> pd.DataFrame:
    # Futures scan risk (from Module 2)
    futures_df  = build_scan_risk_summary()
    futures_df["instrument_type"] = "FUTURE"
    futures_df["SOM ($)"]         = 0.0
    futures_df["SOM_Binding"]     = False
    futures_df = futures_df.rename(columns={"Scan_Risk ($)": "Final_Risk ($)"})

    # Options scan risk
    opt_rows = []
    for oid in OPTION_PORTFOLIO:
        r = compute_option_scan_risk(oid)
        spec = OPTION_SPECS[oid]
        opt_rows.append({
            "Contract"        : oid,
            "Net_Pos"         : r["net_position"],
            "Final_Risk ($)"  : r["final_risk"],
            "Worst_Scenario"  : r["worst_scenario"],
            "Worst_Price_Move": None,
            "Worst_PnL ($)"   : r["worst_pnl"],
            "instrument_type" : f"OPT-{r['type']}",
            "SOM ($)"         : r["som"],
            "SOM_Binding"     : r["som_binding"],
        })
    options_df = pd.DataFrame(opt_rows)

    combined = pd.concat([futures_df, options_df], ignore_index=True)
    return combined


# =============================================================================
# 6.7 PRINT FUNCTIONS
# =============================================================================

def print_option_greeks_table():
    print(f"\n{'='*80}")
    print("MODULE 6 — OPTION POSITIONS: PRICING & GREEKS")
    print(f"{'='*80}")
    print(f"\n  {'OptionID':<18} {'Type':<6} {'Strike':>8} {'Money':>5} "
          f"{'Pos':>5} {'Premium':>9} {'Delta':>8} {'Gamma':>8} {'Vega':>8}")
    print(f"  {'-'*76}")
    for oid in OPTION_PORTFOLIO:
        r    = compute_option_scan_risk(oid)
        spec = OPTION_SPECS[oid]
        print(f"  {oid:<18} {r['type']:<6} {spec.strike:>8.2f} {r['moneyness']:>5} "
              f"{r['net_position']:>5} {r['current_premium']:>9.4f} "
              f"{r['delta']:>8.4f} {r['gamma']:>8.4f} {r['vega']:>8.4f}")
    print(f"{'='*80}")


def print_option_scan_risk_table():
    print(f"\n{'='*80}")
    print("MODULE 6 — OPTION SCAN RISK + SHORT OPTION MINIMUM")
    print(f"{'='*80}")
    print(f"\n  {'OptionID':<18} {'Pos':>5} {'ScanRisk':>10} {'SOM':>10} "
          f"{'SOMBinding':>11} {'FinalRisk':>11} {'WorstScen':>10}")
    print(f"  {'-'*76}")
    for oid in OPTION_PORTFOLIO:
        r = compute_option_scan_risk(oid)
        binding_str = "*** YES ***" if r["som_binding"] else "no"
        print(f"  {oid:<18} {r['net_position']:>5} {r['scan_risk']:>10,.2f} "
              f"{r['som']:>10,.2f} {binding_str:>11} "
              f"{r['final_risk']:>11,.2f} {'S'+str(r['worst_scenario']):>10}")
    print(f"{'='*80}")


def print_combined_scan_risk(combined_df: pd.DataFrame):
    print(f"\n{'='*80}")
    print("MODULE 6 — COMBINED SCAN RISK (FUTURES + OPTIONS)")
    print(f"{'='*80}")
    print(f"\n  {'Contract':<20} {'Type':<12} {'Net_Pos':>8} "
          f"{'FinalRisk ($)':>14} {'SOM ($)':>10} {'SOM_Binding':>12}")
    print(f"  {'-'*76}")
    for _, row in combined_df.iterrows():
        binding_str = "YES" if row["SOM_Binding"] else "-"
        print(f"  {row['Contract']:<20} {row['instrument_type']:<12} "
              f"{int(row['Net_Pos']):>8} {row['Final_Risk ($)']:>14,.2f} "
              f"{row['SOM ($)']:>10,.2f} {binding_str:>12}")
    print(f"  {'-'*76}")
    print(f"  {'TOTAL GROSS RISK':<44} "
          f"{combined_df['Final_Risk ($)'].sum():>14,.2f}")
    print(f"{'='*80}")


def print_option_scenario_table(option_id: str):
    df   = compute_option_scenario_pnl(option_id)
    spec = OPTION_SPECS[option_id]
    pos  = OPTION_PORTFOLIO[option_id]
    direction = "LONG" if pos["net_position"] > 0 else "SHORT"

    print(f"\n{'='*85}")
    print(f"SCENARIO P&L — {option_id}  |  {direction} {abs(pos['net_position'])} lots  "
          f"|  Strike: {spec.strike}  |  IV: {spec.implied_vol*100:.0f}%")
    print(f"{'='*85}")
    print(f"{'Scen':<5} {'PFrac':>7} {'Vol':>6} {'Wt':>5} {'dF':>8} "
          f"{'HypoF':>9} {'HypoVol':>8} {'CurrPrem':>9} {'ScenPrem':>9} "
          f"{'RawPnL':>11} {'WtdPnL':>11}")
    print("-" * 85)
    for _, row in df.iterrows():
        print(f"{int(row.Scenario):<5} {row.Price_Frac:>7.4f} {row.Vol_Shift:>6} "
              f"{row.Weight:>5.2f} {row.Price_Move:>8.4f} {row.Hypo_Futures:>9.4f} "
              f"{row.Hypo_Vol:>8.4f} {row.Current_Premium:>9.4f} "
              f"{row.Scenario_Premium:>9.4f} {row.Raw_PnL:>11,.2f} "
              f"{row.Weighted_PnL:>11,.2f}")
    print(f"{'='*85}")


# =============================================================================
# 6.8 EXTENDED run_span_model() — futures + options
# =============================================================================

def run_span_model_v2():
    """
    Full SPAN model with options.
    Sections 1–5 unchanged. Section 6 adds options layer.
    """
    # --- Run base model first ---
    print_run_metadata()
    print_position_summary()

    scan_summary = build_scan_risk_summary()
    spreading    = compute_total_spreading(scan_summary)
    margin_df    = compute_span_margins(scan_summary, spreading)
    port_summary = compute_portfolio_margin(margin_df, spreading)

    scenario_df  = build_scenario_summary()
    waterfall_df = build_margin_waterfall(spreading, port_summary)

    print_scenario_summary(scenario_df)
    print_spreading_summary(spreading)
    print_per_position_margin(margin_df)
    print_margin_waterfall(waterfall_df, port_summary)

    # --- Section 6: Options layer ---
    print(f"\n{'─'*75}")
    print("  SECTION 6 — OPTIONS LAYER")
    print(f"{'─'*75}")

    print_option_greeks_table()
    print_option_scan_risk_table()

    # Scenario tables for each option
    for oid in OPTION_PORTFOLIO:
        print_option_scenario_table(oid)

    # Combined scan risk
    combined_df = build_combined_scan_risk()
    print_combined_scan_risk(combined_df)

    # Portfolio totals with options
    futures_margin  = port_summary["portfolio_span_margin"]
    options_risk    = sum(
        compute_option_scan_risk(oid)["final_risk"] for oid in OPTION_PORTFOLIO
    )
    total_margin    = futures_margin + options_risk

    print(f"\n{'='*55}")
    print("  FULL PORTFOLIO MARGIN (FUTURES + OPTIONS)")
    print(f"{'='*55}")
    print(f"  Futures SPAN Margin  : ${futures_margin:>12,.2f}")
    print(f"  Options Risk (gross) : ${options_risk:>12,.2f}")
    print(f"  {'─'*42}")
    print(f"  TOTAL MARGIN REQ.    : ${total_margin:>12,.2f}")
    print(f"{'='*55}")

    print_footer()

    return {
        "scan_summary"  : scan_summary,
        "spreading"     : spreading,
        "margin_df"     : margin_df,
        "port_summary"  : port_summary,
        "combined_df"   : combined_df,
        "total_margin"  : total_margin,
    }


if __name__ == "__main__":
    results = run_span_model_v2()


═══════════════════════════════════════════════════════════════════════════
═══════════════════ CME SPAN DUMMY MODEL — FULL REPORT ════════════════════
═══════════════════════════════════════════════════════════════════════════

  Run Timestamp  : 2026-03-19 08:31:57
  Model Version  : SPAN (Futures + Calendar/Inter-Commodity Spreads)
  Positions      : 4 contracts across 3 products
  Scenario Grid  : 16 scenarios (±1/3, ±2/3, ±1x, ±2x scan range + extremes)

───────────────────────────────────────────────────────────────────────────
  SECTION 1 — PORTFOLIO POSITIONS
───────────────────────────────────────────────────────────────────────────

  Contract     Dir     Lots      Entry   Settlement   PointVal Group     
  ----------------------------------------------------------------------
  CL_Mar25     LONG       5     78.500       79.200       1000 ENERGY    
  CL_Jun25     SHORT      3     79.800       80.100       1000 ENERGY    
  NG_Mar25     LONG       4      2.650        2.710  

/tmp/ipykernel_239/1432314680.py:333: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([futures_df, options_df], ignore_index=True)
